# XGBoost V3 vs Logistic Regression Baseline -- Model Comparison Analysis

This notebook compares two pixel-level fire spread prediction models trained on V3 pipeline data
(prev_fire_state approach -- no temporal leakage by construction):

| Model | F1 | Precision | Recall | AUC-ROC |
|-------|-----|-----------|--------|--------|
| **XGBoost V3** | 0.884 | 0.822 | 0.956 | 0.995 |
| **LogReg V3** (baseline) | 0.832 | 0.731 | 0.966 | 0.994 |

Both models use identical feature sets (80 features) and the same train/test split
(8 train fires, 4 test fires). XGBoost improves F1 by +5.2pp over the linear baseline,
almost entirely through better precision (+9.1pp) while maintaining comparable recall.

**Data**: V3 pipeline -- `prev_fire_state[t] = labels[t-1]` pre-shifted by the data pipeline.
No confidence values leak into features. Target is `labels[t]`.

**Test fires**: CampFire, CreekFire, DolanFire, GlassFire

---
## 1. Setup

In [ ]:
%matplotlib inline

import sys
import json
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO_ROOT = Path(".").resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

from data.pipeline_loader import (
    load_processed_fire_data,
    iter_pixel_samples_v3,
    get_pixel_feature_names_v3,
    subsample_negatives,
)

# Paths
CHECKPOINT_DIR = REPO_ROOT / "data" / "checkpoints"
ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
XGB_RESULTS_PATH = ANALYSIS_DIR / "xgboost_v3" / "results.json"
LR_RESULTS_PATH = ANALYSIS_DIR / "logreg_v3" / "results.json"
XGB_CHECKPOINT = CHECKPOINT_DIR / "xgboost_v3_best.pkl"
LR_CHECKPOINT = CHECKPOINT_DIR / "logreg_v3_best.pkl"

# Pipeline data (optional -- for live inference)
_CANDIDATE_PIPELINE_DIRS = [
    REPO_ROOT.parent / "wildfire-data-pipeline" / "data",
    Path.home() / "Desktop" / "Current Projects" / "wildfire-data-pipeline" / "data",
]
PIPELINE_DIR = None
for _cand in _CANDIDATE_PIPELINE_DIRS:
    if _cand.resolve().is_dir():
        PIPELINE_DIR = _cand.resolve()
        break

# Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

SEQ_LEN = 6
FEATURE_NAMES = get_pixel_feature_names_v3(seq_len=SEQ_LEN)

print(f"Repo root:     {REPO_ROOT}")
print(f"Pipeline data: {PIPELINE_DIR or 'NOT FOUND (will use results JSON)'}")
print(f"Feature count: {len(FEATURE_NAMES)}")

---
## 2. Load Models and Results

In [ ]:
# Load results JSON
with open(XGB_RESULTS_PATH) as f:
    xgb_results = json.load(f)
with open(LR_RESULTS_PATH) as f:
    lr_results = json.load(f)

print("XGBoost V3 results loaded.")
print(f"  Train fires: {xgb_results['train_fires']}")
print(f"  Test fires:  {xgb_results['test_fires']}")
print(f"  Train samples: {xgb_results['train_samples']:,}")
print(f"  Test samples:  {xgb_results['test_samples']:,}")
print(f"  Train time:    {xgb_results['train_time_s']:.1f}s")
print()
print("LogReg V3 results loaded.")
print(f"  Train time:    {lr_results['train_time_s']:.1f}s")

In [ ]:
# Load model checkpoints
xgb_model = None
lr_model = None
lr_scaler = None

try:
    with open(XGB_CHECKPOINT, "rb") as f:
        xgb_model = pickle.load(f)
    print(f"XGBoost model loaded: {type(xgb_model).__name__}")
    print(f"  n_estimators={xgb_model.n_estimators}, max_depth={xgb_model.max_depth}")
    print(f"  n_features_in_={getattr(xgb_model, 'n_features_in_', 'unknown')}")
except Exception as e:
    print(f"Could not load XGBoost checkpoint: {e}")

try:
    with open(LR_CHECKPOINT, "rb") as f:
        lr_bundle = pickle.load(f)
    lr_model = lr_bundle["model"]
    lr_scaler = lr_bundle["scaler"]
    print(f"LogReg model loaded: {type(lr_model).__name__}")
    print(f"  n_features_in_={getattr(lr_model, 'n_features_in_', 'unknown')}")
    print(f"  Scaler: {type(lr_scaler).__name__}")
except Exception as e:
    print(f"Could not load LogReg checkpoint: {e}")

---
## 3. Overall Metrics Comparison

XGBoost V3 achieves **F1=0.884** vs LogReg's **F1=0.832** -- a +5.2pp improvement.
The gain comes almost entirely from precision (+9.1pp), while recall stays comparable
(XGBoost 0.956 vs LogReg 0.966). Both models have extremely high AUC (>0.994),
indicating strong discriminative ability -- the difference is in where each model
places its default decision boundary.

In [ ]:
metrics = ["F1", "Precision", "Recall", "AUC-ROC"]
keys = ["f1", "precision", "recall", "auc_roc"]

xgb_vals = [xgb_results[k] for k in keys]
lr_vals = [lr_results[k] for k in keys]

# Print table
print(f"{'Metric':<12} {'XGBoost V3':>12} {'LogReg V3':>12} {'Delta':>10}")
print("-" * 48)
for name, xv, lv in zip(metrics, xgb_vals, lr_vals):
    delta = xv - lv
    sign = "+" if delta >= 0 else ""
    print(f"{name:<12} {xv:>12.4f} {lv:>12.4f} {sign}{delta:>9.4f}")

# Bar chart
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, xgb_vals, width, label="XGBoost V3", color="#2196F3", edgecolor="white")
bars2 = ax.bar(x + width / 2, lr_vals, width, label="LogReg V3", color="#FF9800", edgecolor="white")

ax.set_ylabel("Score")
ax.set_title("Overall Metrics: XGBoost V3 vs LogReg V3 Baseline")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0.65, 1.02)
ax.legend()

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Per-Fire Metrics

Performance broken down by test fire. XGBoost consistently outperforms LogReg across all
four test fires, with the largest F1 gap on **DolanFire** (+12.3pp) and **GlassFire** (+8.9pp).
The improvement is smallest on **CampFire** (+3.4pp), which is also the fire with the
fewest positive samples.

In [ ]:
test_fires = xgb_results["test_fires"]

# Print per-fire comparison
print(f"{'Fire':<20} {'XGB F1':>8} {'LR F1':>8} {'XGB P':>8} {'LR P':>8} {'XGB R':>8} {'LR R':>8}")
print("-" * 72)
for fire in test_fires:
    xf = xgb_results["per_fire_test"][fire]
    lf = lr_results["per_fire_test"][fire]
    print(f"{fire:<20} {xf['f1']:>8.3f} {lf['f1']:>8.3f} "
          f"{xf['precision']:>8.3f} {lf['precision']:>8.3f} "
          f"{xf['recall']:>8.3f} {lf['recall']:>8.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_labels = ["F1", "Precision", "Recall"]
metric_keys = ["f1", "precision", "recall"]

for ax, mlabel, mkey in zip(axes, metric_labels, metric_keys):
    xgb_fire_vals = [xgb_results["per_fire_test"][f][mkey] for f in test_fires]
    lr_fire_vals = [lr_results["per_fire_test"][f][mkey] for f in test_fires]

    x = np.arange(len(test_fires))
    width = 0.35

    bars1 = ax.bar(x - width / 2, xgb_fire_vals, width, label="XGBoost", color="#2196F3", edgecolor="white")
    bars2 = ax.bar(x + width / 2, lr_fire_vals, width, label="LogReg", color="#FF9800", edgecolor="white")

    ax.set_title(f"Per-Fire {mlabel}")
    ax.set_xticks(x)
    ax.set_xticklabels([f.replace("Fire", "") for f in test_fires], rotation=15)
    ax.set_ylabel(mlabel)

    if mkey == "recall":
        ax.set_ylim(0.90, 1.0)
    else:
        ax.set_ylim(0.55, 1.0)

    # Value labels
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

    ax.legend(fontsize=9)

fig.suptitle("Per-Fire Metrics: XGBoost V3 vs LogReg V3", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Feature Importance (XGBoost -- Gain)

XGBoost feature importance by gain reveals a clear hierarchy:

1. **`prev_fire_t-0_di0_dj0`** (0.375) -- the most recent fire state at the pixel itself. Fire persistence is the single strongest predictor.
2. **`prev_fire_t-1_di0_dj0`** (0.234) -- fire state one hour earlier at the same pixel.
3. **`prev_fire_neighborhood`** (0.230) -- fraction of 3x3 neighborhood that was burning.

These top 3 features account for ~84% of total gain. The model is primarily learning
"fire persists and spreads locally" -- exactly what physics tells us.

Weather and terrain features (gust, tmp, fuel_load, slope) contribute small but
nonzero gain, suggesting they help resolve edge cases where fire state alone is ambiguous.

In [ ]:
# XGBoost feature importance from results JSON
feat_imp = xgb_results["feature_importance"]

# Sort and take top 20
sorted_feats = sorted(feat_imp.items(), key=lambda x: x[1], reverse=True)
top_n = 20
top_names = [f[0] for f in sorted_feats[:top_n]][::-1]
top_vals = [f[1] for f in sorted_feats[:top_n]][::-1]

# Color by feature category
def get_color(name):
    if "prev_fire" in name or "distance" in name or "neighborhood" in name:
        return "#E53935"  # Fire features: red
    elif any(w in name for w in ["tmp", "gust", "ugrd", "vgrd", "dpt", "soil"]):
        return "#1E88E5"  # Hourly weather: blue
    elif any(w in name for w in ["erc", "bi", "fm100", "fm1000", "vpd"]):
        return "#43A047"  # Daily fire weather: green
    elif any(w in name for w in ["slope", "aspect", "elevation", "tpi", "fuel", "canopy"]):
        return "#8E24AA"  # Terrain: purple
    elif any(w in name for w in ["hour", "doy"]):
        return "#FB8C00"  # Temporal: orange
    elif any(w in name for w in ["NDVI", "EVI"]):
        return "#00897B"  # Vegetation: teal
    return "#757575"

colors = [get_color(n) for n in top_names]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(range(top_n), top_vals, color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_names, fontsize=9)
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("XGBoost V3 -- Top 20 Features by Gain")

# Value labels
for bar, val in zip(bars, top_vals):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#E53935", label="Fire state"),
    Patch(facecolor="#1E88E5", label="Hourly weather"),
    Patch(facecolor="#43A047", label="Daily fire weather"),
    Patch(facecolor="#8E24AA", label="Terrain"),
    Patch(facecolor="#FB8C00", label="Temporal"),
    Patch(facecolor="#00897B", label="Vegetation"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

# Summary stats
fire_gain = sum(v for k, v in feat_imp.items() if "prev_fire" in k or "distance" in k or "neighborhood" in k)
total_gain = sum(feat_imp.values())
print(f"\nFire-state features: {fire_gain:.3f} / {total_gain:.3f} = {fire_gain/total_gain:.1%} of total gain")

---
## 6. Feature Importance (Logistic Regression -- |Coefficients|)

Logistic regression coefficients (standardized) show which features have the strongest
linear relationship with fire probability. Unlike XGBoost, the linear model cannot
learn complex interactions, so the coefficient magnitudes reflect marginal importance
after accounting for correlations.

In [ ]:
if lr_model is not None:
    coefs = np.abs(lr_model.coef_[0])
    sorted_idx = np.argsort(coefs)[::-1]

    top_n = 20
    top_lr_names = [FEATURE_NAMES[i] if i < len(FEATURE_NAMES) else f"feat_{i}" for i in sorted_idx[:top_n]][::-1]
    top_lr_vals = [coefs[i] for i in sorted_idx[:top_n]][::-1]

    # Signed coefficients for the top features (to see direction)
    raw_coefs = lr_model.coef_[0]
    top_lr_signed = [raw_coefs[i] for i in sorted_idx[:top_n]][::-1]

    colors_lr = ["#E53935" if v > 0 else "#1E88E5" for v in top_lr_signed]

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(range(top_n), top_lr_vals, color=colors_lr, edgecolor="white", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_lr_names, fontsize=9)
    ax.set_xlabel("|Coefficient| (standardized features)")
    ax.set_title("LogReg V3 -- Top 20 Features by |Coefficient|")

    for bar, val in zip(bars, top_lr_vals):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8)

    legend_elements = [
        Patch(facecolor="#E53935", label="Positive coef (increases fire prob)"),
        Patch(facecolor="#1E88E5", label="Negative coef (decreases fire prob)"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()

    # Print top 10 with signs
    print("\nTop 10 LogReg features (signed):")
    for rank, idx in enumerate(sorted_idx[:10], 1):
        nm = FEATURE_NAMES[idx] if idx < len(FEATURE_NAMES) else f"feat_{idx}"
        print(f"  {rank:>2}. {nm:<35} coef={raw_coefs[idx]:>+.4f} (|{coefs[idx]:.4f}|)")
else:
    print("LogReg model not loaded -- skipping coefficient analysis.")

---
## 7. Threshold Sweep

Both models output probabilities. The default threshold is 0.5, but fire prediction
often benefits from tuning this. We sweep thresholds from 0.1 to 0.9 and plot
Precision-Recall curves and F1 vs threshold.

If pipeline data is available, we load a test fire for live inference.
Otherwise, we use the confusion matrix counts from results JSON to compute
the default-threshold operating point and show approximate curves.

In [ ]:
# Attempt to load pipeline data for live threshold sweep
HAS_LIVE_DATA = False
y_test_all = None
xgb_probs = None
lr_probs = None

if PIPELINE_DIR is not None and xgb_model is not None and lr_model is not None:
    # Pick the smallest test fire for speed
    test_fire_sizes = {}
    for fire in xgb_results["test_fires"]:
        pf = xgb_results["per_fire_test"][fire]
        test_fire_sizes[fire] = pf.get("n_pos", 0)
    chosen_fire = min(test_fire_sizes, key=test_fire_sizes.get)

    try:
        processed, features = load_processed_fire_data(chosen_fire, str(PIPELINE_DIR))
        X_list, y_list = [], []
        for fv, label, _w in iter_pixel_samples_v3(processed, features, seq_len=SEQ_LEN):
            X_list.append(fv)
            y_list.append(label)
        X_live = np.array(X_list, dtype=np.float32)
        y_test_all = np.array(y_list, dtype=np.float32)

        xgb_probs = xgb_model.predict_proba(X_live)[:, 1]
        X_live_scaled = np.nan_to_num(lr_scaler.transform(X_live), nan=0.0, posinf=0.0, neginf=0.0)
        lr_probs = lr_model.predict_proba(X_live_scaled)[:, 1]

        HAS_LIVE_DATA = True
        n_pos = int((y_test_all > 0).sum())
        print(f"Live data loaded: {chosen_fire} ({len(y_test_all):,} samples, {n_pos:,} positive)")
    except Exception as e:
        print(f"Could not load live data for {chosen_fire}: {e}")
        print("Falling back to results JSON.")

if not HAS_LIVE_DATA:
    print("No live pipeline data available -- threshold sweep uses results JSON operating point.")

In [ ]:
if HAS_LIVE_DATA:
    thresholds = np.arange(0.05, 0.96, 0.05)

    def sweep(y_true, y_prob, thresholds):
        results = {"precision": [], "recall": [], "f1": []}
        for t in thresholds:
            y_pred = (y_prob >= t).astype(int)
            tp = ((y_pred == 1) & (y_true == 1)).sum()
            fp = ((y_pred == 1) & (y_true == 0)).sum()
            fn = ((y_pred == 0) & (y_true == 1)).sum()
            p = tp / (tp + fp) if (tp + fp) > 0 else 0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
            results["precision"].append(p)
            results["recall"].append(r)
            results["f1"].append(f1)
        return results

    xgb_sweep = sweep(y_test_all, xgb_probs, thresholds)
    lr_sweep = sweep(y_test_all, lr_probs, thresholds)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Precision-Recall curve
    ax1.plot(xgb_sweep["recall"], xgb_sweep["precision"], "o-", color="#2196F3",
             label="XGBoost V3", markersize=4, linewidth=2)
    ax1.plot(lr_sweep["recall"], lr_sweep["precision"], "s-", color="#FF9800",
             label="LogReg V3", markersize=4, linewidth=2)

    # Mark default threshold (0.5)
    idx_05 = list(thresholds).index(0.5) if 0.5 in thresholds else np.argmin(np.abs(thresholds - 0.5))
    ax1.plot(xgb_sweep["recall"][idx_05], xgb_sweep["precision"][idx_05], "*",
             color="#2196F3", markersize=15, zorder=5)
    ax1.plot(lr_sweep["recall"][idx_05], lr_sweep["precision"][idx_05], "*",
             color="#FF9800", markersize=15, zorder=5)

    ax1.set_xlabel("Recall")
    ax1.set_ylabel("Precision")
    ax1.set_title("Precision-Recall Curve (stars = threshold 0.5)")
    ax1.legend()
    ax1.set_xlim(0, 1.05)
    ax1.set_ylim(0, 1.05)

    # F1 vs Threshold
    ax2.plot(thresholds, xgb_sweep["f1"], "o-", color="#2196F3",
             label="XGBoost V3", markersize=4, linewidth=2)
    ax2.plot(thresholds, lr_sweep["f1"], "s-", color="#FF9800",
             label="LogReg V3", markersize=4, linewidth=2)

    # Mark optimal thresholds
    xgb_best_idx = np.argmax(xgb_sweep["f1"])
    lr_best_idx = np.argmax(lr_sweep["f1"])
    ax2.axvline(thresholds[xgb_best_idx], color="#2196F3", linestyle="--", alpha=0.5)
    ax2.axvline(thresholds[lr_best_idx], color="#FF9800", linestyle="--", alpha=0.5)
    ax2.plot(thresholds[xgb_best_idx], xgb_sweep["f1"][xgb_best_idx], "*",
             color="#2196F3", markersize=15, zorder=5)
    ax2.plot(thresholds[lr_best_idx], lr_sweep["f1"][lr_best_idx], "*",
             color="#FF9800", markersize=15, zorder=5)

    ax2.set_xlabel("Threshold")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("F1 vs Decision Threshold (stars = optimal)")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    print(f"XGBoost optimal threshold: {thresholds[xgb_best_idx]:.2f} (F1={xgb_sweep['f1'][xgb_best_idx]:.4f})")
    print(f"LogReg  optimal threshold: {thresholds[lr_best_idx]:.2f} (F1={lr_sweep['f1'][lr_best_idx]:.4f})")

else:
    # Fallback: show operating point from results JSON
    fig, ax = plt.subplots(figsize=(7, 5))

    # XGBoost default operating point
    ax.plot(xgb_results["recall"], xgb_results["precision"], "*",
            color="#2196F3", markersize=18, label=f"XGBoost (F1={xgb_results['f1']:.3f})")
    # LogReg default operating point
    ax.plot(lr_results["recall"], lr_results["precision"], "*",
            color="#FF9800", markersize=18, label=f"LogReg (F1={lr_results['f1']:.3f})")

    # Per-fire operating points
    for fire in xgb_results["test_fires"]:
        xf = xgb_results["per_fire_test"][fire]
        lf = lr_results["per_fire_test"][fire]
        label = fire.replace("Fire", "")
        ax.plot(xf["recall"], xf["precision"], "o", color="#2196F3", markersize=8, alpha=0.6)
        ax.plot(lf["recall"], lf["precision"], "s", color="#FF9800", markersize=8, alpha=0.6)
        # Annotate XGB points
        ax.annotate(label, (xf["recall"], xf["precision"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8, color="#2196F3")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Operating Points (threshold=0.5)")
    ax.set_xlim(0.90, 1.0)
    ax.set_ylim(0.55, 0.95)
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("Note: Live threshold sweep requires pipeline data. Showing default operating points only.")

---
## 8. Precision-Recall Tradeoff Analysis

Both models exhibit **recall-heavy default behavior**: at threshold=0.5, XGBoost achieves
R=0.956 but P=0.822, and LogReg achieves R=0.966 but P=0.731. This is partly by design
(class weighting / `scale_pos_weight` pushes the model to avoid missing fires) and partly
because the fire class is relatively rare in the pixel population.

**When would you want to shift the threshold?**

- **Lower threshold (higher recall)**: Emergency response / evacuation decisions where
  missing a fire pixel is catastrophic. Accept more false positives to ensure no real
  fire goes undetected. Already near saturation at default.

- **Higher threshold (higher precision)**: Resource allocation (e.g., positioning fire
  crews) where acting on false positives wastes limited resources. Raising the threshold
  to 0.6-0.7 could substantially reduce false alarms while keeping recall above 0.90.

- **F1-optimal threshold**: Often near 0.4-0.5 for these models. The default is already
  close to the F1 peak.

The key insight: XGBoost's advantage over LogReg is most visible in precision. At matched
recall levels, XGBoost produces significantly fewer false alarms -- the nonlinear
interactions it learns (e.g., fire state + wind direction + terrain slope) allow it to
reject spurious fire predictions that the linear model cannot distinguish.

In [ ]:
# Visualize the precision gap at matched recall
fig, ax = plt.subplots(figsize=(10, 4))

fires = xgb_results["test_fires"]
x = np.arange(len(fires))
width = 0.35

# Precision values
xgb_prec = [xgb_results["per_fire_test"][f]["precision"] for f in fires]
lr_prec = [lr_results["per_fire_test"][f]["precision"] for f in fires]

bars1 = ax.bar(x - width / 2, xgb_prec, width, label="XGBoost Precision", color="#2196F3", edgecolor="white")
bars2 = ax.bar(x + width / 2, lr_prec, width, label="LogReg Precision", color="#FF9800", edgecolor="white")

# Annotate the gap
for i, (xp, lp) in enumerate(zip(xgb_prec, lr_prec)):
    gap = xp - lp
    mid = (xp + lp) / 2
    ax.annotate(f"+{gap:.2f}", xy=(i, mid), fontsize=10, ha="center",
                fontweight="bold", color="#333")

ax.set_ylabel("Precision")
ax.set_title("Precision Gap: XGBoost vs LogReg (both at default threshold=0.5)")
ax.set_xticks(x)
ax.set_xticklabels([f.replace("Fire", "") for f in fires])
ax.set_ylim(0.5, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Confusion Matrices

Side-by-side confusion matrices for XGBoost and LogReg, aggregated across all test fires.
LogReg results JSON includes tp/fp/fn counts directly. For XGBoost, we compute them
from per-fire data if available, otherwise estimate from precision/recall/n_pos.

In [ ]:
def compute_confusion_from_results(results):
    """Extract or estimate TP, FP, FN, TN from results dict."""
    # Check if top-level counts exist
    if "tp" in results and "fp" in results and "fn" in results:
        tp = results["tp"]
        fp = results["fp"]
        fn = results["fn"]
        total = results.get("test_samples", tp + fp + fn)
        tn = total - tp - fp - fn
        return tp, fp, fn, tn

    # Sum per-fire counts if available
    tp_total, fp_total, fn_total = 0, 0, 0
    has_counts = True
    for fire, data in results["per_fire_test"].items():
        if "tp" in data:
            tp_total += data["tp"]
            fp_total += data["fp"]
            fn_total += data["fn"]
        else:
            has_counts = False
            break

    if has_counts:
        total = results.get("test_samples", tp_total + fp_total + fn_total)
        tn_total = total - tp_total - fp_total - fn_total
        return tp_total, fp_total, fn_total, tn_total

    # Estimate from precision, recall, and total n_pos
    n_pos = sum(d["n_pos"] for d in results["per_fire_test"].values())
    r = results["recall"]
    p = results["precision"]
    tp = int(round(n_pos * r))
    fn = n_pos - tp
    fp = int(round(tp / p - tp)) if p > 0 else 0
    total = results.get("test_samples", n_pos * 10)
    tn = total - tp - fp - fn
    return tp, fp, fn, tn


xgb_tp, xgb_fp, xgb_fn, xgb_tn = compute_confusion_from_results(xgb_results)
lr_tp, lr_fp, lr_fn, lr_tn = compute_confusion_from_results(lr_results)

print(f"XGBoost: TP={xgb_tp:,} FP={xgb_fp:,} FN={xgb_fn:,} TN={xgb_tn:,}")
print(f"LogReg:  TP={lr_tp:,}  FP={lr_fp:,}  FN={lr_fn:,}  TN={lr_tn:,}")

In [ ]:
def plot_confusion_matrix(ax, tp, fp, fn, tn, title, cmap="Blues"):
    cm = np.array([[tn, fp], [fn, tp]])
    total = cm.sum()

    im = ax.imshow(cm, interpolation="nearest", cmap=cmap, aspect="auto")

    # Labels with counts and percentages
    for i in range(2):
        for j in range(2):
            val = cm[i, j]
            pct = val / total * 100
            color = "white" if val > cm.max() * 0.5 else "black"
            ax.text(j, i, f"{val:,}\n({pct:.1f}%)",
                    ha="center", va="center", fontsize=11, color=color, fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred: No Fire", "Pred: Fire"])
    ax.set_yticklabels(["Actual: No Fire", "Actual: Fire"])
    ax.set_title(title, fontsize=12)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_confusion_matrix(ax1, xgb_tp, xgb_fp, xgb_fn, xgb_tn,
                      f"XGBoost V3 (F1={xgb_results['f1']:.3f})", cmap="Blues")
plot_confusion_matrix(ax2, lr_tp, lr_fp, lr_fn, lr_tn,
                      f"LogReg V3 (F1={lr_results['f1']:.3f})", cmap="Oranges")

plt.suptitle("Confusion Matrices (All Test Fires)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-fire confusion matrices (from LogReg results which have counts; estimate XGBoost)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, fire in enumerate(xgb_results["test_fires"]):
    xf = xgb_results["per_fire_test"][fire]
    lf = lr_results["per_fire_test"][fire]

    # LogReg has counts; XGBoost needs estimation
    lr_tp_f = lf["tp"]
    lr_fp_f = lf["fp"]
    lr_fn_f = lf["fn"]

    # Estimate XGBoost counts from precision/recall/n_pos
    n_pos_f = xf["n_pos"]
    xgb_tp_f = int(round(n_pos_f * xf["recall"]))
    xgb_fn_f = n_pos_f - xgb_tp_f
    xgb_fp_f = int(round(xgb_tp_f / xf["precision"] - xgb_tp_f)) if xf["precision"] > 0 else 0

    # Estimate total samples per fire from LogReg (which has tp+fp+fn)
    total_f = lr_tp_f + lr_fp_f + lr_fn_f + (xgb_results["test_samples"] // len(xgb_results["test_fires"]))
    # Better: use test_samples proportionally by n_pos
    total_n_pos = sum(d["n_pos"] for d in lr_results["per_fire_test"].values())
    total_samples = lr_results["test_samples"]
    # Approximate: ratio of this fire's data to total
    fire_total = int(total_samples * (lr_tp_f + lr_fp_f + lr_fn_f) /
                     sum(d["tp"] + d["fp"] + d["fn"] for d in lr_results["per_fire_test"].values()))

    xgb_tn_f = max(fire_total - xgb_tp_f - xgb_fp_f - xgb_fn_f, 0)
    lr_tn_f = max(fire_total - lr_tp_f - lr_fp_f - lr_fn_f, 0)

    short_name = fire.replace("Fire", "")
    plot_confusion_matrix(axes[0, col], xgb_tp_f, xgb_fp_f, xgb_fn_f, xgb_tn_f,
                          f"XGB: {short_name}", cmap="Blues")
    plot_confusion_matrix(axes[1, col], lr_tp_f, lr_fp_f, lr_fn_f, lr_tn_f,
                          f"LR: {short_name}", cmap="Oranges")

plt.suptitle("Per-Fire Confusion Matrices (XGBoost top row, LogReg bottom)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Model Comparison Summary

### Key Takeaways

1. **XGBoost V3 is the clear winner** for overall quality: F1=0.884 vs 0.832, with the
   improvement driven by +9.1pp in precision. It makes far fewer false fire predictions.

2. **Both models share the same recall-heavy bias**: both achieve >0.95 recall at default
   threshold. For fire prediction this is appropriate -- missing a real fire is worse than
   a false alarm.

3. **Feature importance confirms physical intuition**: the top 3 features (same-pixel fire
   state at t-0 and t-1, plus neighborhood fire fraction) account for ~84% of XGBoost's
   gain. Fire spread is primarily a local persistence + neighborhood contagion process.

4. **LogReg's main limitation is false positives**: the linear model overpredicts fire,
   especially on DolanFire (P=0.618 vs XGBoost's 0.823). It cannot learn the nonlinear
   interactions that help XGBoost reject false alarms.

5. **Training speed**: LogReg trains in ~1.9s vs XGBoost's ~30s. For rapid iteration
   during feature engineering, LogReg is a useful fast baseline.

### When to Use Which Model

| Scenario | Recommended Model |
|----------|------------------|
| Production fire prediction | XGBoost V3 |
| Quick feature engineering experiments | LogReg V3 (15x faster) |
| Maximum recall (evacuation alerts) | Either (both >0.95 recall) |
| High-precision resource allocation | XGBoost V3 with threshold >0.5 |
| Interpretability / auditing | LogReg V3 (linear coefficients) |